In [1]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets


  Cloning https://github.com/unslothai/unsloth.git to c:\users\a.subramanyam\appdata\local\temp\2\pip-install-_wnirzwk\unsloth_73e247f7464d428593a0b6fbea4976e2


  ERROR: Error [WinError 2] The system cannot find the file specified while executing command git version
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?

[notice] A new release of pip is available: 23.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/208.7 MB ? eta -:--:--
    --------------------------------------- 2.9/208.7 MB 94.1 MB/s eta 0:00:03
   - -------------------------------------- 6.5/208.7 MB 82.6 MB/s eta 0:00:03
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208.7 MB 101.3 MB/s eta 0:00:02
   - -------------------------------------- 9.4/208


[notice] A new release of pip is available: 23.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Hardware-specific parameters for NVIDIA T4 (G4dn)
max_seq_length = 2048         # Sufficient for extensive SAP BYD resolution steps
dtype = torch.float16         # T4 requires FP16 (bfloat16 is NOT supported)
load_in_4bit = True           # Mandatory for fitting 8B weights in 16 GB VRAM

# Load base model & tokenizer
# (Note: If using Qwen2.5-7B/8B, simply change the model path to "unsloth/Qwen2.5-7B-bnb-4bit")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)



[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\LLM\Python3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1017 15:33:35.125000 14720 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1017 15:33:35.253000 14720 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.6: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.835 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.14.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 399/399 [00:04<00:00, 83.36it/s]


In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                    # Rank 16 provides balanced accuracy for enterprise KBAs
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,                          # 0 is optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",    # Essential to stay under 16 GB VRAM
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)



Unsloth 2026.10.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: Offloading embeddings to RAM to save 1.16 GB.


In [4]:
#This prompt template incorporates domain context for SAP Business ByDesign support and maps the "response" key from your attached JSON file (file_1792217468283491.json):

# Domain-specific prompt templates for SAP Business ByDesign KBAs
alpaca_prompt_with_input = """Below is an instruction describing an SAP Business ByDesign issue or task, paired with an input providing contextual screen/system details. Write a resolution that accurately answers the request based on official KBA guidance.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

alpaca_prompt_no_input = """Below is an instruction describing an SAP Business ByDesign issue or task. Write a resolution that accurately answers the request based on official KBA guidance.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Prevents infinite repetitive token generation

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    # Reads 'response' (or falls back to 'output' if key differs)
    responses    = examples.get("response", examples.get("output", []))
    
    texts = []
    for instruction, inp, response in zip(instructions, inputs, responses):
        if inp and str(inp).strip():
            text = alpaca_prompt_with_input.format(instruction, inp, response) + EOS_TOKEN
        else:
            text = alpaca_prompt_no_input.format(instruction, response) + EOS_TOKEN
        texts.append(text)
        
    return {"text": texts}

# Load your local SAP BYD JSON dataset
dataset = load_dataset("json", data_files="file_1792230788342029.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded and formatted {len(dataset)} SAP BYD KBA records.")


Generating train split: 99 examples [00:00, 5465.21 examples/s]
Map: 100%|██████████| 99/99 [00:00<00:00, 3710.65 examples/s]

Loaded and formatted 99 SAP BYD KBA records.


In [5]:

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Set to True if KBA articles are uniformly short (<500 tokens)
    args=TrainingArguments(
        per_device_train_batch_size=1,       # Keeps VRAM peak < 12 GB on T4
        gradient_accumulation_steps=8,       # Effective batch size of 8
        warmup_steps=10,
        max_steps=80,                        # Adjust based on dataset volume or use num_train_epochs=3
        learning_rate=2e-4,
        fp16=True,                           # REQUIRED for NVIDIA T4 (Turing)
        bf16=False,                          # MUST BE FALSE (T4 does not support BF16)
        logging_steps=5,
        optim="adamw_8bit",                  # Saves ~2 GB VRAM compared to adamw_torch
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="sap_byd_qwen_outputs",
        report_to="none",
    ),
)


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"]: 100%|██████████| 99/99 [00:00<00:00, 396.21 examples/s]


In [6]:
# Train the model and display loss curves
trainer_stats = trainer.train()
print("Training Complete. Final loss metrics:", trainer_stats.metrics)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 99 | Num Epochs = 7 | Total steps = 80
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.675337


KeyboardInterrupt: 

In [ ]:
# Activate Unsloth native 2x generation speedup
FastLanguageModel.for_inference(model)

test_instruction = "You are Unable to Find The Desired Extension field in the Adaptation Mode for Arithmetic Calculation"
test_input = "You have created few extension fields on the Purchase Order screen under Adaptation mode and want to add/create a calculated field using those extension fields but you are unable to find the desired extension fields for calculation."

prompt = alpaca_prompt_with_input.format(test_instruction, test_input, "")

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.3,  # Lower temperature produces more deterministic, factual SAP steps
    top_p=0.9,
)

print("--- Model Output ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Model Output ---
Below is an instruction describing an SAP Business ByDesign issue or task, paired with an input providing contextual screen/system details. Write a resolution that accurately answers the request based on official KBA guidance.

### Instruction:
You are Unable to Find The Desired Extension field in the Adaptation Mode for Arithmetic Calculation

### Input:
You have created few extension fields on the Purchase Order screen under Adaptation mode and want to add/create a calculated field using those extension fields but you are unable to find the desired extension fields for calculation.

### Response:
Go to the Business Configuration work center.
Go to the Implementation Projects view.
Select the relevant project.
Select the relevant activity.
Select the relevant task.
Select the relevant subtask.
Select the relevant screen.
Select the relevant field.
Select the relevant action.
Select the relevant option.
Select the relevant field.
Select the relevant option.
Select

In [8]:
# 1. Save LoRA adapter locally
model.save_pretrained("sap_byd_qwen3_lora")
tokenizer.save_pretrained("sap_byd_qwen3_lora")

# 2. (Optional) Export to GGUF format to serve with Ollama / vLLM locally:
# model.save_pretrained_gguf("sap_byd_qwen3_gguf", tokenizer, quantization_method="q4_k_m")
print("LoRA adapters saved successfully to 'sap_byd_qwen3_lora'.")


Unsloth: Restored added_tokens_decoder metadata in sap_byd_qwen3_lora\tokenizer_config.json.


LoRA adapters saved successfully to 'sap_byd_qwen3_lora'.
